# Agentic Workflow Discovery — Demo

This notebook walks through the full pipeline: generate synthetic data → clean → cluster → train → predict.

In [ ]:
from scripts.generate_mock_data import generate_dataset
from agentic_workflow_discovery.pipeline.orchestrator import Orchestrator
from agentic_workflow_discovery.graph.metrics import evaluate_partition
import matplotlib.pyplot as plt
import numpy as np

## 1. Generate synthetic data

In [ ]:
events, labels = generate_dataset(n_sessions=10, seed=42)
print(f"Generated {len(events)} events across {len(set(e.user_id for e in events))} sessions")

## 2. Ingestion & Cleaning

In [ ]:
orch = Orchestrator(context_length=5, embedding_dim=32, hidden_dim=64, n_layers=1)
orch.ingest(events)
print(f"Train: {len(orch.split.train)} | Val: {len(orch.split.val)} | Test: {len(orch.split.test)}")

## 3. Graph & Clustering

In [ ]:
orch.build_graph(n_clusters=3)
metrics = evaluate_partition(orch.graph, orch.state_to_cluster)
print(f"Graph nodes: {orch.graph.number_of_nodes()}")
print(f"Edges: {orch.graph.number_of_edges()}")
print(f"Silhouette: {metrics['silhouette']:.3f}")
print(f"Clusters found: {int(metrics['n_clusters'])}")

## 4. Train GRU Model

In [ ]:
losses = orch.train_model(vocab_size=200, max_epochs=20, batch_size=16, early_stop_patience=10)
print(f"Final training loss: {losses[-1]:.4f}")
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()

## 5. Predict Next Action

In [ ]:
context = orch.split.test[:5]
if len(context) == 5:
    result = orch.predict(context)
    print(f"Predicted token ID: {result['token_id']}  (confidence: {result['confidence']:.2%})")
    print("Top-3 candidates:")
    for item in result['top_k']:
        print(f"  {item['token_id']:4d}  {item['score']:.2%}  {item['state']}")